# GPU Roofline Analysis — CUDA SGEMM Kernels
**Goal**: classify each kernel as memory-bound or compute-bound by plotting
arithmetic intensity (AI = FLOPs/byte) vs attainable performance (TFLOP/s).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path

plt.rcParams.update({'figure.dpi': 140, 'font.size': 11})


## 1. Hardware rooflines
Values for H100 SXM5 (adjust for your GPU).

In [ ]:
# H100 SXM5 specs
PEAK_FP32_TFLOPS   = 66.9   # theoretical FP32 TFLOP/s
PEAK_TC_FP16_TFLOPS = 989.4 # tensor core FP16 TFLOP/s
PEAK_MEM_BW_TBs    = 3.35   # HBM3 memory bandwidth TB/s

# Ridge points: AI where compute-bound and memory-bound lines intersect
ridge_fp32 = PEAK_FP32_TFLOPS / PEAK_MEM_BW_TBs
ridge_tc   = PEAK_TC_FP16_TFLOPS / PEAK_MEM_BW_TBs
print(f'FP32 ridge point: {ridge_fp32:.1f} FLOPs/byte')
print(f'TC   ridge point: {ridge_tc:.1f} FLOPs/byte')


## 2. Measured kernel data
Replace with actual `ncu` CSV values from `scripts/run_ncu.sh`.

In [ ]:
# Kernel measurements (M=N=K=4096, FP32)
# AI = 2*M*N*K / (bytes_read + bytes_written)
kernels = pd.DataFrame({
    'name':   ['naive', 'tiled_32x32', 'vectorized', 'tensor_core'],
    'ai':     [0.5,     8.3,           31.7,         63.2],        # FLOPs/byte
    'perf':   [1.4,     12.8,          36.9,         55.1],        # TFLOP/s (FP32-eq)
    'color':  ['#e74c3c','#e67e22','#3498db','#2ecc71'],
})


## 3. Roofline plot

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

ai_range = np.logspace(-1, 3, 400)

# Memory-bound slope
mem_roof = PEAK_MEM_BW_TBs * ai_range

# FP32 roofline
fp32_roof = np.minimum(PEAK_FP32_TFLOPS, mem_roof)
ax.loglog(ai_range, fp32_roof, 'b-', lw=2, label='FP32 CUDA cores')

# Tensor core roofline
tc_roof = np.minimum(PEAK_TC_FP16_TFLOPS, mem_roof)
ax.loglog(ai_range, tc_roof, 'g--', lw=2, label='FP16 Tensor Cores')

# Ridge point markers
ax.axvline(ridge_fp32, color='b', alpha=0.3, lw=1, linestyle=':')
ax.axvline(ridge_tc,   color='g', alpha=0.3, lw=1, linestyle=':')

# Plot kernels
for _, row in kernels.iterrows():
    ax.scatter(row.ai, row.perf, color=row.color, s=120, zorder=5)
    ax.annotate(row['name'], (row.ai, row.perf),
                xytext=(8, 4), textcoords='offset points', fontsize=9)

ax.set_xlabel('Arithmetic Intensity (FLOPs / byte)', fontsize=12)
ax.set_ylabel('Performance (TFLOP/s)', fontsize=12)
ax.set_title('Roofline Model — H100 SXM5 | SGEMM 4096×4096', fontsize=13)
ax.legend()
ax.grid(True, which='both', alpha=0.3)
ax.set_xlim(0.1, 1000)
ax.set_ylim(0.1, 2000)
plt.tight_layout()
plt.savefig('roofline_h100.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: roofline_h100.png')


## 4. Interpretation
| Kernel | AI | Regime | Bottleneck |
|--------|----|--------|------------|
| naive | 0.5 | Memory-bound | L2/HBM bandwidth |
| tiled | 8.3 | Memory-bound | Still below FP32 ridge |
| vectorized | 31.7 | Approaching ridge | float4 LDS reduces traffic |
| tensor_core | 63.2 | Compute-bound | MMA throughput limit |

The naive kernel achieves <5% of peak — each global load feeds only one FMA.
Tiling moves data into shared memory (re-use factor ≈ TILE), closing the gap.
Tensor cores shift the ridge to ~300 FLOPs/byte, making most workloads memory-bound again at small batch sizes.